# MiniTest

## Objetivo

Traducir instrucciones secuenciales en español a scripts para automatizar tareas básicas en el navegador

## Alcance

* Abrir páginas web mediante su URL
* Escribir texto dentro de campos (como usuarios, contraseñas o barras de búsqueda)
* Simular clics en la pantalla
* Ejecutar las acciones de forma secuencial (línea por línea)

## Especificaciones léxicas

| Token      | Lexema (Ejemplos)                 | Patrón (Expresión Regular)  |
|------------|-----------------------------------|-----------------------------|
| `ABRIR`    | ABRIR                             | `ABRIR`                     |
| `ESCRIBIR` | ESCRIBIR                          | `ESCRIBIR`                  |
| `EN`       | EN                                | `EN`                        |
| `CLIC`     | CLIC                              | `CLIC`                      |
| `ID`       | ID                                | `ID`                        |
| `XPATH`    | XPATH                             | `XPATH`                     |
| `CADENA`   | "google.com", "admin", "Ingresar" | `\"[^\"]*\"`                |

## Especificaciones sintácticas

```text
<Programa> ::= <Instruccion> | <Instruccion> <Programa>
<Instruccion> ::= ABRIR CADENA 
<Instruccion> ::= ESCRIBIR CADENA EN <Selector> CADENA 
<Instruccion> ::= CLIC <Selector> CADENA
<Selector> ::= ID | XPATH
```

## Especificaciones semánticas

| Regla | Acciones Semánticas |
| :--- | :--- |
| `<Programa> ::= <Instruccion>` | Traduce una sola instrucción a código Python/Selenium. |
| `<Programa> ::= <Instruccion> <Programa>` | Concatena múltiples líneas de código Python/Selenium generadas. |
| `<Instruccion> ::= ABRIR CADENA` | **Trad:** Genera el código para que el driver navegue a la URL indicada (ej: `driver.get(CADENA)`). |
| `<Instruccion> ::= ESCRIBIR CADENA EN <Selector> CADENA` | **Trad:** Genera el código para buscar el elemento web según el `<Selector>` y le envía las teclas de la primera CADENA (ej: `element.send_keys(CADENA)`). |
| `<Instruccion> ::= CLIC <Selector> CADENA` | **Trad:** Genera el código para buscar el elemento web según el `<Selector>` indicado y simula un clic (ej: `element.click()`). |
| `<Selector> ::= ID \| XPATH` | Define el método de búsqueda en Selenium (`By.ID` o `By.XPATH`) que utilizarán las acciones de clic y escritura. |


## Implementación

1. **Lenguaje base:** Python
2. **Metacompilador:** PLY (`ply.lex` y `ply.yacc`)
3. **Ejecución de LLM:** Ollama

In [26]:
import ply.lex as lex
import ply.yacc as yacc
import sys

# Creé una variable '__file__' falsa al entorno para que PLY funcione sin problemas.
# Me tiraba errores en la notebook de jupyter al no encontrar el archivo .py
if not hasattr(sys.modules['__main__'], '__file__'):
    sys.modules['__main__'].__file__ = 'tp_traductor_falso.py'

# Use una clase para aislar las reglas del PLY, evitando que lea basura del entorno de jupyter. 
class TraductorWeb:
    # 1. ANALIZADOR LÉXICO
    tokens = (
        'ABRIR', 'ESCRIBIR', 'EN', 'CLIC', 'ID', 'XPATH', 'CADENA'
    )

    t_ABRIR    = r'ABRIR'
    t_ESCRIBIR = r'ESCRIBIR'
    t_EN       = r'EN'
    t_CLIC     = r'CLIC'
    t_ID       = r'ID'
    t_XPATH    = r'XPATH'

    def t_CADENA(self, t): #Detecta cualquier texto escrito entre ''
        r'\"[^\"]*\"'
        return t

    t_ignore = ' \t' #Ignora espacios en blanco y tabulaciones.

    def t_newline(self, t): #Cuenta los saltos de linea. 
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t): #Error si escriben un Token que no existe. 
        print(f"Error léxico: palabra no reconocida '{t.value[0]}' en línea {t.lexer.lineno}")
        t.lexer.skip(1)

    # 2. ANALIZADOR SINTÁCTICO Y SEMÁNTICO
    # Permite que el programa acepte varias instrucciones.
    def p_programa_multiple(self, p):   
        '''programa : instruccion programa'''
        p[0] = p[1] + "\n" + p[2]

    # Regla de corte. Define qué hacer cuando solo hay una instrucción o cuando es la última línea del programa.
    def p_programa_simple(self, p):
        '''programa : instruccion'''
        p[0] = p[1]

    # Traduce el comando ABRIR.
    def p_instruccion_abrir(self, p):
        '''instruccion : ABRIR CADENA'''
        p[0] = f"driver.get({p[2]})"

    # Traduce el comando ESCRIBIR.
    def p_instruccion_escribir(self, p):
        '''instruccion : ESCRIBIR CADENA EN selector CADENA'''
        p[0] = f"driver.find_element({p[4]}, {p[5]}).send_keys({p[2]})"

    # Traduce el comando CLIC.
    def p_instruccion_clic(self, p):
        '''instruccion : CLIC selector CADENA'''
        p[0] = f"driver.find_element({p[2]}, {p[3]}).click()"

    # Traduce el ID.
    def p_selector_id(self, p):
        '''selector : ID'''
        p[0] = "By.ID"

    # Traduce el XPATH.
    def p_selector_xpath(self, p):
        '''selector : XPATH'''
        p[0] = "By.XPATH"

    # Se ejecuta con errores como (ej: ABRIR ABRIR).
    def p_error(self, p):
        if p:
            print(f"Error de sintaxis cerca de '{p.value}'")
        else:
            print("Error de sintaxis al final del código")

    # 3. CONSTRUCTOR
    def __init__(self):
        # Usamos object=self para encapsular el analizador en esta clase
        self.lexer = lex.lex(object=self)
        self.parser = yacc.yacc(module=self, write_tables=False)
        
    #Función principal, recibe el texto en español como string, lo pasa por el lexer y el parser, retorna el código final traducido.
    def traducir(self, codigo):
        return self.parser.parse(codigo, lexer=self.lexer)

## Casos de Prueba (TestSuite)

In [30]:
mi_traductor = TraductorWeb()
print("       INICIANDO TEST SUITE (10 CASOS)  ")

# Caso número 1: Instrucción válida simple
print("\n# CASO 1: Válido - Solo ABRIR")
caso1 = 'ABRIR "https://www.github.com"'
print(mi_traductor.traducir(caso1))

# Caso número 2: Instrucción válida de clic con XPATH
print("\n# CASO 2: Válido - CLIC con XPATH")
caso2 = 'CLIC XPATH "//boton-ingresar"'
print(mi_traductor.traducir(caso2))

# Caso número 3: Secuencia completa válida
print("\n# CASO 3: Válido - Secuencia completa (ABRIR, ESCRIBIR, CLIC)")
caso3 = '''ABRIR "https://miapp.com/login"
ESCRIBIR "ulises_admin" EN ID "usuario"
CLIC ID "btn-entrar"'''
print(mi_traductor.traducir(caso3))

# Caso número 4: Válido - Múltiples clics seguidos
print("\n# CASO 4: Válido - Múltiples clics")
caso4 = '''CLIC XPATH "//menu"
CLIC ID "opcion-configuracion"'''
print(mi_traductor.traducir(caso4))

# Caso número 5: Error Léxico - Comando mal escrito
print("\n# CASO 5: Error Léxico - Comando inexistente (ABRRIR)")
caso5 = 'ABRRIR "https://www.google.com"'
mi_traductor.traducir(caso5)

# Caso número 6: Error Léxico - Símbolo no reconocido
print("\n# CASO 6: Error Léxico - Símbolo especial no definido (@)")
caso6 = 'ESCRIBIR "hola" EN @ID "caja"'
mi_traductor.traducir(caso6)

# Caso número 7: Error Sintáctico - Falta la palabra clave EN
print("\n# CASO 7: Error Sintáctico - Falta el nexo 'EN' al escribir")
caso7 = 'ESCRIBIR "mi_clave" ID "password"'
mi_traductor.traducir(caso7)

# Caso número 8: Error Sintáctico - Orden invertido
print("\n# CASO 8: Error Sintáctico - Cadena antes del comando")
caso8 = '"https://www.google.com" ABRIR'
mi_traductor.traducir(caso8)

# Caso número 9: Error Sintáctico - Falta la cadena final en el CLIC
print("\n# CASO 9: Error Sintáctico - CLIC sin especificar el nombre del ID")
caso9 = 'CLIC ID'
mi_traductor.traducir(caso9)

# Caso número 10: Válido - Caso largo simulando automatización de alquileres
print("\n# CASO 10: Válido - Automatización de carga de datos")
caso10 = '''ABRIR "https://sistema-alquileres.com"
ESCRIBIR "Inquilina_Local1" EN ID "cliente"
ESCRIBIR "15000" EN ID "monto_luz"
CLIC XPATH "//btn-guardar"'''
print(mi_traductor.traducir(caso10))

       INICIANDO TEST SUITE (10 CASOS)  

# CASO 1: Válido - Solo ABRIR
driver.get("https://www.github.com")

# CASO 2: Válido - CLIC con XPATH
driver.find_element(By.XPATH, "//boton-ingresar").click()

# CASO 3: Válido - Secuencia completa (ABRIR, ESCRIBIR, CLIC)
driver.get("https://miapp.com/login")
driver.find_element(By.ID, "usuario").send_keys("ulises_admin")
driver.find_element(By.ID, "btn-entrar").click()

# CASO 4: Válido - Múltiples clics
driver.find_element(By.XPATH, "//menu").click()
driver.find_element(By.ID, "opcion-configuracion").click()

# CASO 5: Error Léxico - Comando inexistente (ABRRIR)
Error léxico: palabra no reconocida 'A' en línea 4
Error léxico: palabra no reconocida 'B' en línea 4
Error léxico: palabra no reconocida 'R' en línea 4
Error léxico: palabra no reconocida 'R' en línea 4
Error léxico: palabra no reconocida 'I' en línea 4
Error léxico: palabra no reconocida 'R' en línea 4
Error de sintaxis cerca de '"https://www.google.com"'

# CASO 6: Error Léxico - 

Generating LALR tables


**Integración con Ollama:**

In [31]:
import ollama

# 1. Obtenemos el código Python generado por el traductor en el Caso 10
codigo_final = mi_traductor.traducir(caso10)

# 2. Función para conectar con el LLM local y documentar el script
def explicar_codigo_selenium(codigo):
    print("Consultando a Ollama localmente... (esto puede tardar unos segundos)\n")
    respuesta = ollama.chat(
        model='llama3.2',
        messages=[
            {
                'role': 'user',
                'content': f'''
                Sos un experto en automatización con Python y Selenium.
                Explicame brevemente y en español qué hace este código.
                Código a explicar: {codigo}
                '''
            }
        ]
    )
    # Extraemos únicamente el texto de la respuesta del LLM
    return respuesta['message']['content']

# 3. Ejecución y muestra de resultados
explicacion = explicar_codigo_selenium(codigo_final)

print("=== EXPLICACIÓN GENERADA POR OLLAMA ===")
print(explicacion)

Consultando a Ollama localmente... (esto puede tardar unos segundos)

=== EXPLICACIÓN GENERADA POR OLLAMA ===
¡Claro! Este código utiliza Selenium para automatizar la interacción con un sistema web.

**Explicación breve:**

El código se encarga de:

1. **Establecer la conexión**: `driver.get("https://sistema-alquileres.com")` se utiliza para abrir la página web `https://sistema-alquileres.com` en el navegador.
2. **Encontrar elementos**: `driver.find_element(By.ID, "cliente")` busca un elemento con el identificador "cliente" en la página web y lo selecciona. Lo mismo se hace con los elementos con identificador "monto_luz" y la etiqueta XPATH "//btn-guardar".
3. **Introducir texto**: `send_keys()` se utiliza para introducir texto en los campos seleccionados, en este caso, "Inquilina_Local1" en el campo "cliente" y "15000" en el campo "monto_luz".
4. **Hacer clic**: `click()` se utiliza para hacer clic en el botón con la etiqueta XPATH "//btn-guardar".

En resumen, este código es una sec

**Conclusiones:**


Con este trabajo desarrollé un lenguaje propio usando Python y PLY para traducir comandos simples en español a código de Selenium, con el objetivo de hacer más fácil la navegación web.

Con los casos de prueba comprobé que el traductor responde bien, procesa los comandos correctos y detecta cuando hay errores de tipeo o sintaxis.

Por último, sumé Ollama para que la IA documente la explicación de mi código.

**Implementación con Ollama:**

In [14]:
import ollama

ESPECIFICACIONES_LENGUAJE = """
ASPECTOS LÉXICOS:
- Palabras reservadas (tokens): ABRIR, ESCRIBIR, EN, CLIC, ID, XPATH.
- CADENA: texto encerrado entre comillas dobles ("...").
- Se ignoran espacios en blanco y tabulaciones.

ASPECTOS SINTÁCTICOS:
- programa : instrucciones
- instrucciones : instrucciones instruccion | instruccion
- instruccion : ABRIR CADENA
              | ESCRIBIR CADENA EN ID CADENA
              | ESCRIBIR CADENA EN XPATH CADENA
              | CLIC ID CADENA
              | CLIC XPATH CADENA

ASPECTOS SEMÁNTICOS (reglas de traducción a Python/Selenium):
- ABRIR <url> -> driver.get(<url>)
- ESCRIBIR <texto> EN ID <valor> -> driver.find_element(By.ID, <valor>).send_keys(<texto>)
- ESCRIBIR <texto> EN XPATH <valor> -> driver.find_element(By.XPATH, <valor>).send_keys(<texto>)
- CLIC ID <valor> -> driver.find_element(By.ID, <valor>).click()
- CLIC XPATH <valor> -> driver.find_element(By.XPATH, <valor>).click()
"""


def agente_traductor_ollama(programa_fuente: str) -> str:
  prompt = f"""Actuá como un traductor que ejecuta únicamente las acciones semánticas de un lenguaje.
Tu tarea es traducir el programa de entrada a código Python con Selenium respetando estrictamente estas especificaciones:

{ESPECIFICACIONES_LENGUAJE}

Reglas obligatorias de salida:
- Devolvé ÚNICAMENTE las líneas traducidas de Selenium (driver.get, driver.find_element, etc.).
- NO repitas las instrucciones en español en la salida.
- NO agregues imports, ni inicialización de driver, ni comentarios, ni explicaciones.

Programa de entrada:
{programa_fuente}
"""
  respuesta = ollama.chat(
      model="llama3.2",
      messages=[{"role": "user", "content": prompt}],
      options={"temperature": 0},
  )
  return respuesta["message"]["content"].strip()


programa_fuente = '''
ABRIR "https://sistema-alquileres.com"
ESCRIBIR "Inquilina_Local1" EN ID "cliente"
ESCRIBIR "15000" EN ID "monto_luz"
CLIC XPATH "//btn-guardar"
'''

programa_fuente = agente_traductor_ollama(programa_fuente)
print("Traducción generada por el agente Ollama:\n")
print(programa_fuente)

Traducción generada por el agente Ollama:

```python
driver.get("https://sistema-alquileres.com")
driver.find_element(By.ID, "cliente").send_keys("Inquilina_Local1")
driver.find_element(By.ID, "monto_luz").send_keys("15000")
driver.find_element(By.XPATH, "//btn-guardar").click()
```
